<a href="https://colab.research.google.com/github/bipin-nepular/PHPWord/blob/master/notebooks/piper_multilingual_training_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <font color="ffc800"> **[Piper](https://github.com/rhasspy/piper) training notebook.**
## ![Piper logo](https://contribute.rhasspy.org/img/logo.png)

---

- Notebook made by [rmcpantoja](http://github.com/rmcpantoja)
- Collaborator: [Xx_Nessu_xX](http://github.com/Xx_Nessu_xX)

---

# Notes:

- <font color="orange">**Things in orange mean that they are important.**

# Credits:

* [Feanix-Fyre fork](https://github.com/Feanix-Fyre/piper) with some improvements.
* [Tacotron2 NVIDIA training notebook](https://github.com/justinjohn0306/FakeYou-Tacotron2-Notebook) - Dataset duration snippet.
* [🐸TTS](https://github.com/coqui-ai/TTS) - Resampler and XTTS formater demo.

# <font color="ffc800">🔧 ***First steps.*** 🔧

In [1]:
#@markdown ## <font color="ffc800"> **Google Colab Anti-Disconnect.** 🔌
#@markdown ---
#@markdown #### Avoid automatic disconnection. Still, it will disconnect after <font color="orange">**6 to 12 hours**</font>.

import IPython
js_code = '''
function ClickConnect(){
console.log("Working");
document.querySelector("colab-toolbar-button#connect").click()
}
setInterval(ClickConnect,60000)
'''
display(IPython.display.Javascript(js_code))

<IPython.core.display.Javascript object>

In [2]:
#@markdown ## <font color="ffc800"> **Check GPU type.** 👁️
#@markdown ---
#@markdown #### A higher capable GPU can lead to faster training speeds. By default, you will have a <font color="orange">**Tesla T4**</font>.
!nvidia-smi

Fri Apr 17 16:19:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
#@markdown # <font color="ffc800"> **Mount Google Drive.** 📂
#@markdown ---
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [15]:
#@markdown # <font color="ffc800"> **Install software.** 📦
#@markdown ---
#@markdown ####In this cell the synthesizer and its necessary dependencies to execute the training will be installed. (this may take a while)

# clone:
!git clone -q https://github.com/rmcpantoja/piper
%cd /content/piper/src/python
!wget -q "https://raw.githubusercontent.com/coqui-ai/TTS/dev/TTS/bin/resample.py"

# Install dependencies from requirements.txt first
print("Installing dependencies from requirements.txt...")
!python3 -m pip install -q -r requirements.txt || echo "Failed to install some packages from requirements.txt, checking piper-phonemize specifically."

# Install other packages that might not be in requirements.txt or need explicit upgrade
!python3 -m pip install --upgrade gdown transformers

# Fix for piper-phonemize on Python 3.12 - requires building from source
# If piper-phonemize failed to install, attempt to install Rust and build it.
try:
    import piper_phonemize
    print("piper-phonemize is already installed.")
except ImportError:
    print("piper-phonemize not found for Python 3.12, attempting to install Rust and build from source...")
    # Install Rust
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
    # Add cargo to PATH for the current session (important for pip build process)
    %env PATH=/root/.cargo/bin:$PATH
    # Force pip to build piper-phonemize from source
    !python3 -m pip install --no-binary :all: piper-phonemize
    print("Attempted to build and install piper-phonemize from source.")

!bash build_monotonic_align.sh
# Useful vars:
use_whisper = True
print("Done!")

/bin/bash: line 1: git: command not found
/content/piper/src/python
/bin/bash: line 1: wget: command not found
Installing dependencies from requirements.txt...
/bin/bash: line 1: python3: command not found
Failed to install some packages from requirements.txt, checking piper-phonemize specifically.
/bin/bash: line 1: python3: command not found
piper-phonemize not found for Python 3.12, attempting to install Rust and build from source...
/bin/bash: line 1: curl: command not found
/bin/bash: line 1: sh: command not found
env: PATH=/root/.cargo/bin:$PATH
/bin/bash: line 1: python3: command not found
Attempted to build and install piper-phonemize from source.
/bin/bash: line 1: bash: command not found
Done!


In [5]:
print('Attempting to install pytorch_lightning...')
!python3 -m pip install pytorch-lightning

Attempting to install pytorch_lightning...
/bin/bash: line 1: pip: command not found


In [6]:
print('Installing faster_whisper...')
!python3 -m pip install faster-whisper

Installing faster_whisper...
/bin/bash: line 1: pip: command not found


# <font color="ffc800"> 🤖 ***Training.*** 🤖

In [7]:
import os
import wave
import datetime
import shutil

# This function needs to be robust for any path, so it should take an absolute path
def get_dataset_duration(base_path):
    totalduration = 0
    wav_count = 0
    # Walk through the directory to find all .wav files
    for root, _, files in os.walk(base_path):
        for file_name in files:
            if file_name.endswith(".wav"):
                full_file_path = os.path.join(root, file_name)
                try:
                    with wave.open(full_file_path, "rb") as wave_file:
                        frames = wave_file.getnframes()
                        rate = wave_file.getframerate()
                        if rate > 0: # Avoid division by zero
                            duration = frames / float(rate)
                            totalduration += duration
                        else:
                            print(f"Warning: Skipping {full_file_path} due to zero framerate (framerate: {rate}).")
                    wav_count += 1
                except wave.Error as e:
                    print(f"Error opening WAV file {full_file_path}: {e}")
                except Exception as e:
                    print(f"An unexpected error occurred with {full_file_path}: {e}")
    duration_str = str(datetime.timedelta(seconds=round(totalduration, 0)))
    return wav_count, duration_str

# Change to /content for setting up dataset directory
%cd /content

if not os.path.exists("/content/dataset"):
    os.makedirs("/content/dataset")
if not os.path.exists("/content/dataset/wavs"):
    os.makedirs("/content/dataset/wavs")

#@markdown ### Audio dataset folder path:
dataset_folder_path = "/content/drive/MyDrive/bipin" #@param {type:"string"}
dataset_folder_path = dataset_folder_path.strip()

if not dataset_folder_path:
    raise Exception("You must provide a path to the dataset folder.")
if not os.path.exists(dataset_folder_path):
    raise Exception(f"The path provided '{dataset_folder_path}' does not exist. Please set a valid path.")

# Construct the source path for the 'wav' subdirectory within the user's dataset folder
source_wav_path = os.path.join(dataset_folder_path, "wav")
if not os.path.exists(source_wav_path):
    raise Exception(f"The 'wav' subfolder was not found in '{dataset_folder_path}'. Expected structure: {dataset_folder_path}/wav/...")

# Clear existing /content/dataset/wavs before copying
if os.path.exists("/content/dataset/wavs"):
    shutil.rmtree("/content/dataset/wavs")
os.makedirs("/content/dataset/wavs")

print(f"Copying audio content from '{source_wav_path}' to '/content/dataset/wavs'...")
# Copy contents of source_wav_path into /content/dataset/wavs
# The `cp -a` command copies contents recursively.
!cp -a "{source_wav_path}/." "/content/dataset/wavs"

# Calculate duration for the copied files
audio_count, dataset_dur = get_dataset_duration("/content/dataset/wavs")
print(f"Opened dataset with {audio_count} wavs with duration {dataset_dur}.")

# No need to cd into /content/dataset at the end of this cell, let the next cell handle its own %cd


/content
Copying audio content from '/content/drive/MyDrive/bipin/wav' to '/content/dataset/wavs'...
/bin/bash: line 1: cp: command not found
Opened dataset with 0 wavs with duration 0:00:00.


In [8]:
import os

%cd /content/dataset

# Ensure /content/dataset exists (it should from the previous cell, but good practice)
if not os.path.exists("/content/dataset"):
    os.makedirs("/content/dataset")

#@markdown ### Path to metadata.csv file:
# Assuming the user's folder is /content/drive/MyDrive/bipin and metadata is inside it
metadata_file_path = "/content/drive/MyDrive/bipin/metadata.csv" #@param {type:"string"}
metadata_file_path = metadata_file_path.strip()

if not metadata_file_path:
    raise Exception("You must provide a path to the metadata.csv file.")
if not os.path.exists(metadata_file_path):
    raise Exception(f"The metadata file '{metadata_file_path}' does not exist. Please set a valid path.")

# Copy metadata.csv to /content/dataset/metadata.csv
destination_metadata_path = "/content/dataset/metadata.csv"
print(f"Copying metadata file from '{metadata_file_path}' to '{destination_metadata_path}'...")
!cp "{metadata_file_path}" "{destination_metadata_path}"

# Check if metadata.csv was copied successfully
if not os.path.exists(destination_metadata_path):
    raise Exception(f"Failed to copy metadata.csv from {metadata_file_path} to {destination_metadata_path}")

print("Processing metadata file...")
# Read, modify, and write back metadata.csv
modified_lines = []
with open(destination_metadata_path, 'r', encoding='utf-8') as f:
    for line in f:
        # Example: sentence-collector\100.wav|Text
        # Needs to become: wavs/sentence-collector/100.wav|Text
        parts = line.split('|', 1) # Split only on the first '|'
        if len(parts) == 2:
            audio_path = parts[0].strip()
            text_content = parts[1].strip()
            # Replace backslashes with forward slashes and prepend "wavs/"
            # Ensure only single "wavs/" prefix, so remove if already present at start
            if audio_path.startswith("wavs/") or audio_path.startswith("wavs\\"):
                audio_path = audio_path[5:] # Remove existing "wavs/" or "wavs\"
            processed_audio_path = "wavs/" + audio_path.replace('\\', '/')
            modified_lines.append(f"{processed_audio_path}|{text_content}\n")
        else:
            # Keep original line if not in expected format
            modified_lines.append(line)

with open(destination_metadata_path, 'w', encoding='utf-8') as f:
    f.writelines(modified_lines)

print(f"Metadata file processed and saved at '{destination_metadata_path}'.")

# Disable whisper transcription as metadata is provided
use_whisper = False

%cd ..


/content/dataset
Copying metadata file from '/content/drive/MyDrive/bipin/metadata.csv' to '/content/dataset/metadata.csv'...
/bin/bash: line 1: cp: command not found
Processing metadata file...
Metadata file processed and saved at '/content/dataset/metadata.csv'.
/content


In [9]:
#@markdown # <font color="ffc800"> **3. Preprocess dataset.** 🔄
#@markdown ---
import os
if use_whisper:
    import torch
    from faster_whisper import WhisperModel
    from tqdm import tqdm
    from google import colab

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    def make_dataset(path, language):
        metadata = ""
        text = ""
        files = [f for f in os.listdir(path) if f.endswith(".wav")]
        assert len(files) > 0, "You don't have wavs uploaded either! Please upload at least one zip with the wavs in step 2."
        metadata_file = open(f"{path}/../metadata.csv", "w")
        whisper = WhisperModel("large-v3", device=device, compute_type="float16")
        for audio_file in tqdm(files):
            full_path = os.path.join(path, audio_file)
            segments, _ = whisper.transcribe(full_path, word_timestamps=False, language=language)
            for segment in segments:
                text += segment.text
            text = text.strip()
            text = text.replace('\n', ' ')
            metadata = f"{audio_file}|{text}\n"
            metadata_file.write(metadata)
            text = ""
        colab.files.download(f"{path}/../metadata.csv")
        del whisper
        return True

#@markdown ### First of all, select the language of your dataset.
language = "नेपाली" #@param ["ألعَرَبِي", "Català", "čeština", "Dansk", "Deutsch", "Ελληνικά", "English (British)", "English (U.S.)", "Español (Castellano)", "Español (Latinoamericano)", "Suomi", "Français", "Magyar", "Icelandic", "Italiano", "ქართული", "қазақша", "Lëtzebuergesch", "नेपाली", "Nederlands", "Norsk", "Polski", "Português (Brasil)", "Português (Portugal)", "Română", "Русский", "Српски", "Svenska", "Kiswahili", "Türkçe", "украї́нська", "Tiếng Việt", "简体中文"]
#@markdown ---
# language definition:
languages = {
    "ألعَرَبِي": "ar",
    "Català": "ca",
    "čeština": "cs",
    "Dansk": "da",
    "Deutsch": "de",
    "Ελληνικά": "el",
    "English (British)": "en",
    "English (U.S.)": "en-us",
    "Español (Castellano)": "es",
    "Español (Latinoamericano)": "es-419",
    "Suomi.": "fi",
    "Français": "fr",
    "Magyar": "hu",
    "Icelandic": "is",
    "Italiano": "it",
    "ქართული": "ka",
    "қазақша": "kk",
    "Lëtzembuergesch": "lb",
    "नेपाली": "ne",
    "Nederlands": "nl",
    "Norsk": "nb",
    "Polski": "pl",
    "Português (Brasil)": "pt-br",
    "Português (Portugal)": "pt-pt",
    "Română": "ro",
    "Русский": "ru",
    "Српски": "sr",
    "Svenska": "sv",
    "Kiswahili": "sw",
    "Türkçe": "tr",
    "украї́нська": "uk",
    "Tiếng Việt": "vi",
    "简体中文": "zh"
}

def _get_language(code):
    return languages[code]

final_language = _get_language(language)
#@markdown ### Choose a name for your model:
model_name = "bipin_nepali" #@param {type:"string"}
#@markdown ---
# output:
#@markdown ### Choose the working folder: (recommended to save to Drive)

#@markdown The working folder will be used in preprocessing, but also in training the model.
output_path = "/content/drive/MyDrive/colab/piper" #@param {type:"string"}
output_dir = output_path+"/"+model_name
if not os.path.exists(output_dir):
  os.makedirs(output_dir)
#@markdown ---
#@markdown ### Choose dataset format:
dataset_format = "ljspeech" #@param ["ljspeech", "mycroft"]
#@markdown ---
#@markdown ### Is this a single speaker dataset? Otherwise, uncheck:
single_speaker = True #@param {type:"boolean"}
if single_speaker:
  force_sp = " --single-speaker"
else:
  force_sp = ""
#@markdown ---
#@markdown ### Select the sample rate of the dataset:
sample_rate = "22050" #@param ["16000", "22050"]
#@markdown ---
# creating paths:
if not os.path.exists("/content/audio_cache"):
    os.makedirs("/content/audio_cache")
%cd /content/piper/src/python
#@markdown ### Do you want to train using this sample rate, but your audios don't have it?
#@markdown The resampler helps you do it quickly!
resample = False #@param {type:"boolean"}
if resample:
  !python3 resample.py --input_dir "/content/dataset/wavs" --output_dir "/content/dataset/wavs_resampled" --output_sr {sample_rate} --file_ext "wav"
  !mv /content/dataset/wavs_resampled/* /content/dataset/wavs
#@markdown ---
# check transcription:
if use_whisper:
    print("Transcript file hasn't been uploaded. Transcribing these audios using Whisper...")
    make_dataset("/content/dataset/wavs", final_language[:2])
    print("Transcription done! Pre-processing...")
!python3 -m piper_train.preprocess \
  --language {final_language} \
  --input-dir /content/dataset \
  --cache-dir "/content/audio_cache" \
  --output-dir "{output_dir}" \
  --dataset-name "{model_name}" \
  --dataset-format {dataset_format} \
  --sample-rate {sample_rate} \
  {force_sp}
print("Preprocessing done!")

/content/piper/src/python
/bin/bash: line 1: python: command not found
Preprocessing done!


In [10]:
#@markdown # <font color="ffc800"> **4. Settings.** 🧰
#@markdown ---
import json
import ipywidgets as widgets
from IPython.display import display
from google.colab import output
import os
import re
import glob
#@markdown ### <font color="orange">**Select the action to train this dataset: (READ CAREFULLY)**

#@markdown * The option to <font color="orange">continue a training</font> is self-explanatory. If you've previously trained a model with free colab, your time is up and you're considering training it some more, this is ideal for you. You just have to set the same settings that you set when you first trained this model.
#@markdown * The option to <font color="orange">convert a single-speaker model to a multi-speaker model</font> is self-explanatory, and for this it is important that you have processed a dataset that contains text and audio from all possible speakers that you want to train in your model.
#@markdown * The <font color="orange">finetune</font> option is used to train a dataset using a pretrained model, that is, train on that data. This option is ideal if you want to train a very small dataset (more than five minutes recommended).
#@markdown * The <font color="orange">train from scratch</font> option builds features such as dictionary and speech form from scratch, and this may take longer to converge. For this, hours of audio (8 at least) are recommended, which have a large collection of phonemes.

action = "finetune" #@param ["Continue training", "convert single-speaker to multi-speaker model", "finetune", "train from scratch"]
#@markdown ---
if action == "Continue training":
    checkpoints = glob.glob(f"{output_dir}/lightning_logs/**/checkpoints/last.ckpt", recursive=True)
    if len(checkpoints):
        last_checkpoint = sorted(checkpoints, key=lambda x: int(re.findall(r'version_(\d+)', x)[0]))[-1]
        ft_command = f'--resume_from_checkpoint "{last_checkpoint}" '
        print(f"Continuing {model_name}'s training at: {last_checkpoint}")
    else:
        raise Exception("Training cannot be continued as there is no checkpoint to continue at.")
elif action == "finetune":
    if os.path.exists(f"{output_dir}/lightning_logs/version_0/checkpoints/last.ckpt"):
        raise Exception("Oh no! You have already trained this model before, you cannot choose this option since your progress will be lost, and then your previous time will not count. Please select the option to continue a training.")
    else:
        ft_command = '--resume_from_checkpoint "/content/pretrained.ckpt" '
elif action == "convert single-speaker to multi-speaker model":
    if not single_speaker:
        ft_command = '--resume_from_single_speaker_checkpoint "/content/pretrained.ckpt" '
    else:
        raise Exception("This dataset is not a multi-speaker dataset!")
else:
    ft_command = ""
if action== "convert single-speaker to multi-speaker model" or action == "finetune":
    try:
        with open('/content/piper/notebooks/pretrained_models.json') as f:
            pretrained_models = json.load(f)
        if final_language in pretrained_models:
            models = pretrained_models[final_language]
            model_options = [(model_name, model_name) for model_name, model_url in models.items()]
            model_dropdown = widgets.Dropdown(description = "Choose pretrained model", options=model_options)
            download_button = widgets.Button(description="Download")
            def download_model(btn):
                model_name = model_dropdown.value
                model_url = pretrained_models[final_language][model_name]
                print("\033[93mDownloading pretrained model...")
                if model_url.startswith("1"):
                    !gdown -q "{model_url}" -O "/content/pretrained.ckpt"
                elif model_url.startswith("https://drive.google.com/file/d/"):
                    !gdown -q "{model_url}" -O "/content/pretrained.ckpt" --fuzzy
                else:
                    !wget -q "{model_url}" -O "/content/pretrained.ckpt"
                model_dropdown.close()
                download_button.close()
                output.clear()
                if os.path.exists("/content/pretrained.ckpt"):
                    print("\033[93mModel downloaded!")
                else:
                    raise Exception("Couldn't download the pretrained model!")
            download_button.on_click(download_model)
            display(model_dropdown, download_button)
        else:
            raise Exception(f"There are no pretrained models available for the language {final_language}")
    except FileNotFoundError:
        raise Exception("The pretrained_models.json file was not found.")
else:
    print("\033[93mWarning: this model will be trained from scratch. You need at least 8 hours of data for everything to work decent. Good luck!")
#@markdown ### Choose batch size based on this dataset:
batch_size = 12 #@param {type:"integer"}
#@markdown ---

#@markdown ### Choose the quality for this model:

#@markdown * x-low - 16Khz audio, 5-7M params
#@markdown * medium - 22.05Khz audio, 15-20 params
#@markdown * high - 22.05Khz audio, 28-32M params
quality = "high" #@param ["high", "x-low", "medium"]
#@markdown ---
#@markdown ### For how many epochs to save training checkpoints?
#@markdown The larger your dataset, you should set this saving interval to a smaller value, as epochs can progress longer time.
checkpoint_epochs = 5 #@param {type:"integer"}
#@markdown ---
#@markdown ### Interval to save best k models:
#@markdown Set to 0 if you want to disable saving multiple models. If this is the case, check the checkbox below. If set to 1, models will be saved with the file name epoch=xx-step=xx.ckpt, so you will need to empty Drive's trash every so often.
num_ckpt = 1 #@param {type:"integer"}
#@markdown ---
#@markdown ### Save latest model:
#@markdown This checkbox must be checked if you want to save a single model (last.ckpt). Saving a single model is applied only if num_ckpt is equal to 0. If so, the interval parameter of epochs to save is ignored, since the last model per epoch is saved; also, you won't have to worry about storage. Being equal to 1, last.ckpt will be saved, but another model (model_vVersion.ckpt, the latter takes into account the epoch range you set), so you would have to empty the trash often.

#@markdown **It's not recommended to use this option in extremely small datasets, since by saving the last model each epoch, this process will be very fast and the trainer will not be able to save the complete model, which would result in a corrupt last.ckpt.**
save_last = False # @param {type:"boolean"}
#@markdown ---
#@markdown ### Step interval to generate model samples:
log_every_n_steps = 1000 #@param {type:"integer"}
#@markdown ---
#@markdown ### Training epochs:
max_epochs = 10000 #@param {type:"integer"}
#@markdown ---

Dropdown(description='Choose pretrained model', options=(('Google-medium (fine-tuned)', 'Google-medium (fine-t…

Button(description='Download', style=ButtonStyle())

In [11]:
#@markdown # <font color="ffc800"> **5. Run the TensorBoard extension.** 📈
#@markdown ---
#@markdown The TensorBoard is used to visualize the results of the model while it's being trained such as audio and losses.

%load_ext tensorboard
%tensorboard --logdir {output_dir}

ERROR: Could not find `tensorboard`. Please ensure that your PATH
contains an executable `tensorboard` program, or explicitly specify
the path to a TensorBoard binary by setting the `TENSORBOARD_BINARY`
environment variable.

In [12]:
#@markdown # <font color="ffc800"> **6. Train.** 🏋️‍♂️
#@markdown ---
#@markdown ### Run this cell to train your final model!

#@markdown ---
#@markdown ### <font color="orange">**Disable validation?**
#@markdown By uncheck this checkbox, this will allow to train the full dataset, without using any audio files or examples as a validation set. So, it will not be able to generate audios on the tensorboard while it's training. It is recommended to disable validation on extremely small datasets.
validation = False #@param {type:"boolean"}
if validation:
    validation_split = 0.01
    num_test_examples = 1
else:
    validation_split = 0
    num_test_examples = 0
if not save_last:
    save_last_command = ""
else:
    save_last_command = "--save_last True "
get_ipython().system(f'''
python3 -m piper_train \
--dataset-dir "{output_dir}" \
--accelerator 'gpu' \
--devices 1 \
--batch-size {batch_size} \
--validation-split {validation_split} \
--num-test-examples {num_test_examples} \
--quality {quality} \
--checkpoint-epochs {checkpoint_epochs} \
--num_ckpt {num_ckpt} \
{save_last_command}\
--log_every_n_steps {log_every_n_steps} \
--max_epochs {max_epochs} \
{ft_command}\
--precision 32
''')

/bin/bash: line 2: python: command not found


#  <font color="orange">**Have you finished training and want to test the model?**

* If you want to run this model in any software that Piper integrates or the same Piper app, export your model using the [model exporter notebook](https://colab.research.google.com/github/rmcpantoja/piper/blob/master/notebooks/piper_model_exporter.ipynb)!
* Wait! I want to test this right now before exporting it to the supported format for Piper. Test your generated last.ckpt with [this notebook](https://colab.research.google.com/github/rmcpantoja/piper/blob/master/notebooks/piper_inference_(ckpt).ipynb)!